# SVM分类器对eRisk2018数据集进行分类预测

In [5]:
import numpy as np
import json
from sklearn.metrics import f1_score
from sklearn import svm
from sklearn.svm import SVC
from sklearn.preprocessing import normalize
from sklearn.utils.class_weight import compute_class_weight
from sklearn.datasets import make_classification
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE
import json
from scipy.sparse import csr_matrix
from imblearn.under_sampling import RandomUnderSampler

'''根据SVM对测试集进行预测'''

#   训练集及标签
with open('/home/E22301339/depressionDetection-eRisk2018/processedData/data3.json','r')as f:
    data=json.load(f)
train=data['train']
X=np.array(train)
y=np.array((data['y']))

# 找出元素全为0的行的索引,并移除全为0的行
zero_row_indices = np.where(np.all(X == 0, axis=1))[0]
X = np.delete(X, zero_row_indices, axis=0)
y = np.delete(y, zero_row_indices, axis=0)
X=csr_matrix(X)

#   测试集及标签
test=data['X_test']
X_test=np.array(test)
X_test=csr_matrix(X_test)
y_test=np.array(data['y_test'])

# 创建SMOTE对象,过采样
# sm=SMOTE(sampling_strategy='auto')
# X,y=sm.fit_resample(X,y)

# 欠采样策略
rus = RandomUnderSampler(sampling_strategy='auto')
X, y = rus.fit_resample(X, y)

# 计算类别权重
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y), y=y)
class_weights = {i: class_weights[i] for i in range(len(class_weights))}

result=[]
for C in np.arange(0.01,2,0.01):
    # 定义SVM分类器
    clf = SVC(C=C, class_weight=class_weights, probability=True)
    dict={}
    # 训练模型
    clf.fit(X, y)                                                                   
    # 测试模型
    predicted = clf.predict(X_test)
 
    # 预测概率值
    probabilities = clf.predict_proba(X_test)

    accuracy = 100 * np.sum(predicted == y_test) / len(y_test)
    f1 = f1_score(y_test, predicted)
    print('Accuracy of the model on the test samples: {:.2f} %'.format(accuracy))
    print('F1 score of the model on the test samples: {:.2f}'.format(f1))
    print("================================================")
    dict['f1']=f1
    dict['accuracy']=accuracy
    dict['c']=C
    result.append(dict)
print("================================================")


Accuracy of the model on the test samples: 92.44 %
F1 score of the model on the test samples: 0.45
Accuracy of the model on the test samples: 92.44 %
F1 score of the model on the test samples: 0.45
Accuracy of the model on the test samples: 92.44 %
F1 score of the model on the test samples: 0.45
Accuracy of the model on the test samples: 92.44 %
F1 score of the model on the test samples: 0.47
Accuracy of the model on the test samples: 92.68 %
F1 score of the model on the test samples: 0.49
Accuracy of the model on the test samples: 93.29 %
F1 score of the model on the test samples: 0.55
Accuracy of the model on the test samples: 93.41 %
F1 score of the model on the test samples: 0.57
Accuracy of the model on the test samples: 93.54 %
F1 score of the model on the test samples: 0.58
Accuracy of the model on the test samples: 93.66 %
F1 score of the model on the test samples: 0.61
Accuracy of the model on the test samples: 93.54 %
F1 score of the model on the test samples: 0.60
Accuracy o

# 为特征提取后的数据加上测试集的标签

In [ ]:
#  测试集标签
import json
with open('/home/E22301339/depressionDetection-eRisk2018/processedData/data3.json','r')as f:
    data=json.load(f)
with open ('/home/E22301339/depressionDetection-eRisk2018/features/test-golden-truth-test.txt','r+')as f:
        list_data=[]
        list_total=[]  # 存储positive抑郁的用户id
        for item in f.readlines():
            list_data.append(item.strip())
str=list_data[0][:-1].strip()

for item in list_data:
    if(item[len(item)-1]=='1'):
        list_total.append(item[:-1].strip())
    else:
        break
with open('/home/E22301339/depressionDetection-eRisk2018/processedData/total_user_wordpost2.json','r')as f:
        users=list(json.load(f).keys())
labels=[]
for user in users:
    if user in list_total:
        labels.append(1)
    else:
        labels.append(0)
test_y=labels
data['y_test']=test_y
with open('/home/E22301339/depressionDetection-eRisk2018/processedData/data3.json','w')as f:
    json.dump(data,f)

print("End...")
